In [ ]:
!pip install -q onnx 2>/dev/null
!pip install -q onnxruntime 2>/dev/null
!pip install -q onnx-tool 2>/dev/null

In [ ]:
import sys
sys.path.append("/kaggle/input/competitions/neurogolf-2026/neurogolf_utils")
from neurogolf_utils import *
show_legend()

In [ ]:
task_num = 16
examples = load_examples(task_num)
show_examples(examples['train'] + examples['test'])

In [ ]:
import numpy as np
import onnx
import onnxruntime as ort
from onnx import helper as oh, TensorProto

# Environment Fix
if not hasattr(ort, 'ONNXRuntimeError'):
    class MockError(Exception): pass
    ort.ONNXRuntimeError = MockError

_DATA_TYPE = TensorProto.FLOAT

class NeuroSolver:
    def __init__(self, examples):
        self.train = examples['train']
        self.cmap = {i: i for i in range(10)}
        self.offset = (0, 0)
        self.background_fill = 0
        self.strategy = "IDENTITY"
        self._analyze()

    def _analyze(self):
        # 1. Color Mapping
        temp_cmap = {}
        is_color_swap = True
        for p in self.train:
            inp, out = np.array(p['input']), np.array(p['output'])
            if inp.shape != out.shape:
                is_color_swap = False; break
            for s, d in zip(inp.flatten(), out.flatten()):
                if s in temp_cmap and temp_cmap[s] != d:
                    is_color_swap = False; break
                temp_cmap[int(s)] = int(d)
        
        if is_color_swap and temp_cmap:
            self.cmap = temp_cmap
            self.strategy = "GATHER" if any(k != v for k, v in temp_cmap.items()) else "IDENTITY"
            return

        # 2. Spatial Shifts with Background Detection
        for dy, dx in [(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (1,1)]:
            for bg in range(10):
                match = True
                for p in self.train:
                    inp, out = np.array(p['input']), np.array(p['output'])
                    if inp.shape != out.shape:
                        match = False; break
                    h, w = inp.shape
                    shifted = np.full_like(inp, bg)
                    r_in, c_in = slice(max(0, -dy), min(h, h-dy)), slice(max(0, -dx), min(w, w-dx))
                    r_out, c_out = slice(max(0, dy), min(h, h+dy)), slice(max(0, dx), min(w, w+dx))
                    shifted[r_out, c_out] = inp[r_in, c_in]
                    if not np.array_equal(shifted, out):
                        match = False; break
                if match:
                    self.offset, self.background_fill, self.strategy = (dy, dx), bg, "CONV"
                    return

def build_final_optimized_network(solver, task_num):
    x_info = oh.make_tensor_value_info("input", _DATA_TYPE, [1, 10, None, None])
    y_info = oh.make_tensor_value_info("output", _DATA_TYPE, [1, 10, None, None])
    nodes, initializers = [], []

    if solver.strategy == "IDENTITY":
        nodes.append(oh.make_node("Identity", ["input"], ["output"]))

    elif solver.strategy == "GATHER":
        indices = [solver.cmap.get(i, i) for i in range(10)]
        idx_tensor = oh.make_tensor("idx_val", TensorProto.INT64, [10], indices)
        nodes.append(oh.make_node("Constant", [], ["idx"], value=idx_tensor))
        nodes.append(oh.make_node("Gather", ["input", "idx"], ["output"], axis=1))

    elif solver.strategy == "CONV":
        dy, dx = solver.offset
        k_size, k_half = 3, 1
        w_shape = [10, 1, k_size, k_size]
        
        # Weight Initialization
        weights = np.zeros(w_shape, dtype=np.float32)
        for i in range(10):
            weights[i, 0, k_half + dy, k_half + dx] = 1.0
        initializers.append(oh.make_tensor("W", _DATA_TYPE, w_shape, weights.tobytes(), raw=True))

        # Efficiency Optimization: Only add Bias node if background is non-zero
        conv_inputs = ["input", "W"]
        if solver.background_fill != 0:
            bias_vals = np.zeros(10, dtype=np.float32)
            bias_vals[solver.background_fill] = 1.0
            initializers.append(oh.make_tensor("B", _DATA_TYPE, [10], bias_vals.tobytes(), raw=True))
            conv_inputs.append("B")
        
        nodes.append(oh.make_node("Conv", conv_inputs, ["output"],
                                 kernel_shape=[k_size, k_size], pads=[k_half]*4, group=10))

    graph = oh.make_graph(nodes, f"t_{task_num}", [x_info], [y_info], initializers)
    return oh.make_model(graph, opset_imports=[oh.make_opsetid("", 13)])

def encode_grid(grid):
    h, w = len(grid), len(grid[0])
    t = np.zeros((1, 10, h, w), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, v in enumerate(row):
            if 0 <= v < 10: t[0, int(v), r, c] = 1.0
    return t

# --- EXECUTION ---
for task_num in [16, 337]:
    try:
        examples = load_examples(task_num)
        solver = NeuroSolver(examples)
        network = build_final_optimized_network(solver, task_num)
        verify_network(network, task_num, examples)
        with open(f"task{task_num:03d}.onnx", "wb") as f:
            f.write(network.SerializeToString())
        print(f"Task {task_num} Log: Success ({solver.strategy})")
    except Exception as e:
        print(f"Task {task_num} Log: {e}")

In [ ]:
task_num = 276
examples = load_examples(task_num)
show_examples(examples['train'] + examples['test'])

In [ ]:
import numpy as np
import onnx
import onnxruntime as ort
from onnx import helper as oh, TensorProto

_DATA_TYPE = TensorProto.FLOAT

class NeuroSolver:
    def __init__(self, examples):
        self.train = examples['train']
        self.cmap = self._get_color_map()
        self.offset = self._get_spatial_offset()

    def _get_color_map(self):
        cmap = {}
        for p in self.train:
            inp, out = np.array(p['input']), np.array(p['output'])
            if inp.shape != out.shape: return None
            for s, d in zip(inp.flatten(), out.flatten()):
                s_int, d_int = int(s), int(d)
                if s_int in cmap and cmap[s_int] != d_int: return None
                cmap[s_int] = d_int
        return cmap

    def _get_spatial_offset(self):
        # Explicit 5-point neighborhood check
        for dy, dx in [(0, 0), (-1, 0), (1, 0), (0, -1), (0, 1)]:
            match = True
            for p in self.train:
                inp, out = np.array(p['input']), np.array(p['output'])
                if inp.shape != out.shape: match = False; break
                if not np.array_equal(np.roll(inp, (dy, dx), (0, 1)), out):
                    match = False; break
            if match: return (dy, dx)
        return (0, 0)

def build_titan_vFinal(solver, task_num):
    # Dynamic shapes are the key to bypassing the IndexError
    x_info = oh.make_tensor_value_info("input", _DATA_TYPE, [1, 10, None, None])
    y_info = oh.make_tensor_value_info("output", _DATA_TYPE, [1, 10, None, None])
    
    dy, dx = solver.offset
    # Check if we actually need a color remap
    is_swapping = any(k != v for k, v in solver.cmap.items()) if solver.cmap else False
    
    k_size = 1 if (dy == 0 and dx == 0) else 3
    k_half = k_size // 2
    
    # Force Grouped Conv for 10-parameter efficiency
    groups = 1 if is_swapping else 10
    w_shape = [10, 1 if groups == 10 else 10, k_size, k_size]
    weights = np.zeros(w_shape, dtype=np.float32)
    
    for c_in in range(10):
        c_out = solver.cmap.get(c_in, c_in) if solver.cmap else c_in
        # Kernel placement: (center + shift)
        weights[c_out, 0 if groups == 10 else c_in, k_half + dy, k_half + dx] = 1.0

    w_tensor = oh.make_tensor("W", _DATA_TYPE, w_shape, weights.flatten().tolist())
    
    node = oh.make_node(
        "Conv", ["input", "W"], ["output"],
        kernel_shape=[k_size, k_size],
        # Only pad if k_size > 1 to avoid ONNX validation warnings
        pads=[k_half] * 4 if k_size > 1 else [0] * 4,
        group=groups
    )

    graph = oh.make_graph([node], f"t_{task_num}", [x_info], [y_info], [w_tensor])
    return oh.make_model(graph, opset_imports=[oh.make_opsetid("", 11)])

# --- EXECUTION ---
for task_num in [276, 309]:
    try:
        examples = load_examples(task_num)
        solver = NeuroSolver(examples)
        network = build_titan_vFinal(solver, task_num)
        verify_network(network, task_num, examples)
    except Exception as e:
        print(f"Task {task_num} Error: {e}")

In [ ]:
import os, json, zipfile, math, re, io, hashlib
import numpy as np
import onnx
import onnxruntime as ort
from onnx import shape_inference, helper
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed

# --- 1. CUSTOM GRAPH SURGERY ---
def manual_optimize(raw_bytes):
    """
    Custom 'thinking' optimizer. 
    Manually prunes the graph without external simplifiers.
    """
    try:
        model = onnx.load_model(io.BytesIO(raw_bytes))
        
        # A. Strip all Metadata (Protobuf compatible)
        model.doc_string = ""
        while len(model.metadata_props) > 0: del model.metadata_props[-1]
        
        # B. Identify all used inputs/outputs in the nodes
        used_inputs = set()
        for node in model.graph.node:
            node.doc_string = "" # Clean node metadata
            for inp in node.input:
                used_inputs.add(inp)
        
        # Add graph outputs to the used list so we don't prune them!
        for output in model.graph.output:
            used_inputs.add(output.name)

        # C. Prune Unused Initializers
        # We iterate backwards to safely delete while looping
        i = len(model.graph.initializer) - 1
        while i >= 0:
            init = model.graph.initializer[i]
            if init.name not in used_inputs:
                del model.graph.initializer[i]
            i -= 1

        # D. Prune Unused Value Info
        i = len(model.graph.value_info) - 1
        while i >= 0:
            vi = model.graph.value_info[i]
            if vi.name not in used_inputs:
                del model.graph.value_info[i]
            i -= 1

        # E. Shape Inference (The heartbeat of the model)
        model = shape_inference.infer_shapes(model)
        return model.SerializeToString()
    except Exception:
        return raw_bytes

# --- 2. VALIDATION LOGIC ---
def validate(model_bytes, task_data):
    try:
        sess = ort.InferenceSession(model_bytes, providers=['CPUExecutionProvider'])
        in_name = sess.get_inputs()[0].name
        out_name = sess.get_outputs()[0].name
        
        for split in ['train', 'test']:
            if split not in task_data: continue
            for pair in task_data[split]:
                grid = np.array(pair['input'])
                # Padding/Encoding logic
                inp = np.zeros((1, 10, 30, 30), dtype=np.float32)
                gh, gw = grid.shape
                for v in range(10):
                    inp[0, v, :min(gh,30), :min(gw,30)] = (grid[:min(gh,30), :min(gw,30)] == v)
                
                raw_out = sess.run([out_name], {in_name: inp})[0]
                tgt = np.array(pair['output'])
                th, tw = tgt.shape
                
                pred = np.squeeze(raw_out)
                if pred.ndim == 3 and pred.shape[0] == 10:
                    pred = np.argmax(pred[:, :th, :tw], axis=0)
                elif pred.ndim >= 2:
                    pred = pred[:th, :tw].astype(int)
                else: return False
                
                if not np.array_equal(pred, tgt): return False
        return True
    except:
        return False

# --- 3. MASTER PROCESSING ---
def run_pipeline(tk, raw_bytes, data):
    # Try the optimized version first
    opt_bytes = manual_optimize(raw_bytes)
    if validate(opt_bytes, data):
        m = onnx.load_model(io.BytesIO(opt_bytes))
        params = sum(np.prod(p.dims) for p in m.graph.initializer)
        cost = params + len(m.graph.node) * 10
        return (max(1.0, 25.0 - math.log(max(1, cost))), opt_bytes)
    
    # Fallback to original if surgery failed
    if validate(raw_bytes, data):
        m = onnx.load_model(io.BytesIO(raw_bytes))
        params = sum(np.prod(p.dims) for p in m.graph.initializer)
        cost = params + len(m.graph.node) * 10
        return (max(1.0, 25.0 - math.log(max(1, cost))), raw_bytes)
    
    return None

# --- 4. DISCOVERY & EXECUTION ---
def main():
    # Discovery
    tasks, candidates = {}, defaultdict(list)
    
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            p = Path(root) / f
            
            # 1. Handle JSON task files
            if f.endswith(".json") and "task" in f.lower():
                try:
                    tasks[p.stem + ".onnx"] = json.loads(p.read_text())
                except: continue

            # 2. Handle raw ONNX files
            if f.endswith(".onnx"):
                m = re.search(r'task(\d+)', f, re.I)
                if m: 
                    candidates[f"task{m.group(1).zfill(3)}.onnx"].append(p.read_bytes())

            # 3. Handle ZIP files containing ONNX files
            if zipfile.is_zipfile(p):
                try:
                    with zipfile.ZipFile(p, 'r') as z:
                        for name in z.namelist():
                            if name.endswith(".onnx"):
                                m = re.search(r'task(\d+)', name, re.I)
                                if m:
                                    # Read bytes directly from the zip to avoid extraction
                                    with z.open(name) as zf:
                                        candidates[f"task{m.group(1).zfill(3)}.onnx"].append(zf.read())
                except: continue

    print(f"🛠️ Starting Custom Surgery on {len(candidates)} task groups (including Zips)...")
    
    final_models = {}
    # Use max_workers=os.cpu_count() or let it default to avoid overhead
    with ProcessPoolExecutor() as executor:
        for tk, data in sorted(tasks.items()):
            group = candidates.get(tk, [])
            if not group: continue
            
            # Deduplicate by MD5 to avoid redundant surgery/validation
            unique_raws = {hashlib.md5(b).hexdigest(): b for b in group}
            futures = [executor.submit(run_pipeline, tk, b, data) for b in unique_raws.values()]
            
            best_s, best_b = -1, None
            for f in as_completed(futures):
                res = f.result()
                if res:
                    s, b = res
                    if s > best_s: 
                        best_s, best_b = s, b
            
            if best_b:
                final_models[tk] = best_b
                print(f"⭐ {tk} | Optimized & Validated | Score: {best_s:.4f}")

    # Output to submission.zip
    with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        for k, v in sorted(final_models.items()): 
            zf.writestr(k, v)
    print(f"🏁 Finished: {len(final_models)} models.")

if __name__ == "__main__":
    main()


# Ｈ𝐀𝑷𝑷𝓎 🇰𝗮𝘨𝘨🇱𝖎Ｎɢ 💯

In [ ]:
# Code Credits to Gemini - Vibe Code is Path Forward...